# **`Sentiment shifter`**
This tool modifies the emotional tone of input text. The core objective is to shift the sentiment of a sentence (e.g., from positive to negative) while preserving the original semantic meaning. While minimizing changes to the factual content, the tool replaces specific vocabulary with "emotionally charged" synonyms to alter the reader's perception. The solution leverages a multi-API architecture to process text without requiring heavy local libraries.

The overall result is not as good as i thought it would be, but if nothing else it still uses 2 REST APIs and replaces chosen words with its synonyms :)
# USAGE:
The application is a standalone Python script designed to run in any standard Python environment.


**How to Run:**


1.   Locate the *Configuration Section* at the top of the file.
2.   Modify the `BATCH_MODE` variable to select the input data      you wish to process:  
  *   `BATCH_MODE = 1`: Runs a Simple Batch (Proof of Concept)      with short, clear sentences.
  *   `BATCH_MODE = 2`: Runs an Extended Batch with complex,  longer paragraphs
  *   `BATCH_MODE = 3`: Runs All Tests sequentially.




3.   Run the script. The results, including the transformed text and a summary table of changes, will be displayed in the console.




# DESCRIPTION:
The primary goal of the "Sentiment Shifter" is to automate the process of "spinning" text. It identifies neutral or positive descriptors (adjectives) and actions (verbs) and swaps them for synonyms that carry a darker or more negative connotation. This concept mimics the linguistic techniques often used in biased journalism or creative writing, where word choice dictates the mood.


# IMPLEMENTATION DETAILS:
I chose to connect 2 APIs and 1 external model to create a processing pipeline:
*   [**UDPipe API**](https://lindat.mff.cuni.cz/)
    
    Used for grammar & syntax analysis.
    
    It analyzes the input string to identify (POS), for "filtering" the words that are worth changing (changing words like "the" or "to" is unneccessary)
*   [**Datamuse API**](https://www.datamuse.com/api/)

    Used for synonym generation.
    
    "Source of vocabulary/Vocabulary engine"

    Provides the raw options for the swap.

*   [**AFINN Model**](https://raw.githubusercontent.com/fnielsen/afinn/master/afinn/data/AFINN-111.txt)

    Used for sentiment scoring.
    Static dictionary downloaded at the beggining.
    It contains 2,477 words rated from -5 (Very Negative) to +5 (Very Positive).
    Needed for making decision which synonym to use. It looks at the synonyms from Datamuse and checks their score in this list. It only picks a synonym if its score is lower than the original word (more negative).

`Input text -> UDPipe detects important words -> Datamuse provides synonyms to these words -> AFINN provides confidency about the words to replace -> Replace the words in original text -> Output`

# OUPUT:
The program outputs the results in two parts for every test case:


1.   Pessimized Result: the full reconstructured sentence with the new word choices integrated.
2.   Change Log Table: A formatted table listing every word that was swapped, showing the original word, its sentiment score, the new synonym, and its new score.

  **EXAMPLE OUTPUT:**
```text
Downloading Sentiment Model (AFINN-111)... Done! Loaded 2477 words.
=== RUNNING ALL TESTS ===

--- Test 1: Simple Positive ---

Input: The confident leader presented a bold plan.
Analyzing syntax... Done.
Swapping words... ..

============================================================
PESSIMIZED: The self-assured leader presented a rash plan.
============================================================

+--------------+-------------+-----------------+-------------+
| Original     | Old Score   | New Word        | New Score   |
+--------------+-------------+-----------------+-------------+
| confident    | 2           | self-assured    | 0           |
| bold         | 2           | rash            | -2          |
+--------------+-------------+-----------------+-------------+

--- Test 2: Simple Tech ---

Input: This smart device works fast and efficient.
Analyzing syntax... Done.
Swapping words... .

============================================================
PESSIMIZED: This hurt device works fast and efficient.
============================================================

+--------------+-------------+-----------------+-------------+
| Original     | Old Score   | New Word        | New Score   |
+--------------+-------------+-----------------+-------------+
| smart        | 1           | hurt            | -2          |
+--------------+-------------+-----------------+-------------+

In [ ]:
# Version 4.0 with downloading the AFINN Model for sentiment guessing
# The sentiment shift application
# I think this one is so far the best one!
import urllib.request
import urllib.parse
import json
import time

# --- CONFIGURATION SWITCH ---
# 1 = Simple Batch (Proof of Concept)
# 2 = Extended Batch (Complex inputs)
# 3 = Run All Tests
BATCH_MODE = 1

# --- CONFIGURATION ---
UDPIPE_URL = "https://lindat.mff.cuni.cz/services/udpipe/api/process"
DATAMUSE_URL = "https://api.datamuse.com/words"
AFINN_MODEL_URL = "https://raw.githubusercontent.com/fnielsen/afinn/master/afinn/data/AFINN-111.txt"

# --- TEST DATA ---
BATCH_SIMPLE = [
    {"title": "Simple Positive", "text": "The confident leader presented a bold plan."},
    {"title": "Simple Tech",     "text": "This smart device works fast and efficient."},
    {"title": "Simple Social",   "text": "We are happy to support our friends."}
]

BATCH_COMPLEX = [
    {"title": "Political", "text": "The administration's positive stance toward the federation does not correlate with ideas of a free world."},
    {"title": "Corporate", "text": "We are proud to announce massive growth in our sales department, achieved through strong teamwork."},
    {"title": "Romantic",  "text": "My dearest friend, your bright smile brings joy to my peaceful life."}
]


# Global variable to store the model
SENTIMENT_MODEL = {}

def load_sentiment_model():
    """Downloads the AFINN sentiment lexicon from GitHub."""
    print("Downloading Sentiment Model (AFINN-111)...", end=" ")
    try:
        with urllib.request.urlopen(AFINN_MODEL_URL) as response:
            text_data = response.read().decode('utf-8')

        for line in text_data.split('\n'):
            parts = line.split('\t')
            if len(parts) == 2:
                word, score = parts[0], int(parts[1])
                SENTIMENT_MODEL[word] = score
        print(f"Done! Loaded {len(SENTIMENT_MODEL)} words.")
    except Exception as e:
        print(f"\nError downloading model: {e}")

def get_score(word):
    """Returns the score from the downloaded model, or 0 if unknown."""
    return SENTIMENT_MODEL.get(word.lower(), 0)

def api_post(url, data_dict):
    """Helper for POST requests"""
    try:
        data = urllib.parse.urlencode(data_dict).encode('utf-8')
        req = urllib.request.Request(url, data=data)
        with urllib.request.urlopen(req, timeout=10) as response:
            return json.loads(response.read().decode('utf-8'))
    except: return None

def api_get(url, params):
    """Helper for GET requests"""
    try:
        query = urllib.parse.urlencode(params)
        with urllib.request.urlopen(f"{url}?{query}", timeout=5) as response:
            return json.loads(response.read().decode('utf-8'))
    except: return None

def match_suffix(original, new_word):
    """Matches simple verb endings."""
    if original.endswith('s') and not new_word.endswith('s'):
        return new_word + 's'
    if original.endswith('ed') and not new_word.endswith('ed'):
        return new_word + 'ed'
    if original.endswith('ing') and not new_word.endswith('ing'):
        if new_word.endswith('e'): return new_word[:-1] + 'ing'
        return new_word + 'ing'
    return new_word

def print_table(changes):
    """Draws a nice ASCII table without external libraries."""
    '"" Before i used a pretty table but i wanted to omit""'
    if not changes:
        return

    # Define headers and column widths
    headers = ["Original", "Old Score", "New Word", "New Score"]
    widths = [12, 11, 15, 11]

    # Helper to format a row
    def format_row(cols):
        return "| " + " | ".join(f"{str(c):<{w}}" for c, w in zip(cols, widths)) + " |"

    # Create separator line
    separator = "+-" + "-+-".join("-" * w for w in widths) + "-+"

    print("\n" + separator)
    print(format_row(headers))
    print(separator)

    for row in changes:
        print(format_row(row))

    print(separator)

def run_pessimizer(text):
    # 1. Ensure Model is Loaded
    if not SENTIMENT_MODEL:
        load_sentiment_model()

    print(f"\nInput: {text}")
    print("Analyzing syntax...", end=" ")

    # 2. UDPipe Analysis
    udpipe_resp = api_post(UDPIPE_URL, {
        'tokenizer': '', 'tagger': '', 'parser': '',
        'model': 'english-ewt-ud-2.10-220711', 'data': text
    })

    if not udpipe_resp or 'result' not in udpipe_resp:
        print("UDPipe failed.")
        return
    print("Done.")

    # 3. Processing
    lines = udpipe_resp['result'].split('\n')
    result_tokens = []
    changes = [] # Store changes here for the table

    TARGETS = ['ADJ', 'VERB']

    print("Swapping words...", end=" ")

    for line in lines:
        if not line or line.startswith('#'): continue
        parts = line.split('\t')
        if len(parts) < 10: continue
        if '-' in parts[0]: continue

        word, lemma, pos, misc = parts[1], parts[2], parts[3], parts[9]
        final_word = word

        if pos in TARGETS and len(word) > 3:
            current_score = get_score(lemma)

            if current_score > -2:
                # Rate limiting: pause briefly for the free API
                time.sleep(0.1)
                dm_data = api_get(DATAMUSE_URL, {'ml': lemma, 'max': 15})

                if dm_data:
                    best_syn = None
                    lowest_score = current_score

                    for item in dm_data:
                        syn = item['word']
                        if ' ' in syn or syn == lemma: continue

                        syn_score = get_score(syn)

                        if syn_score < lowest_score:
                            lowest_score = syn_score
                            best_syn = syn

                    if best_syn:
                        final_word = match_suffix(word, best_syn)
                        print(".", end="", flush=True) # Progress dot

                        # Add to our changes list
                        changes.append([word, current_score, final_word, lowest_score])

        has_space = "SpaceAfter=No" not in misc
        result_tokens.append((final_word, has_space))

    # 4. Output Text
    output = ""
    for w, space in result_tokens:
        output += w + (" " if space else "")

    print("\n\n" + "="*60)
    print(f"PESSIMIZED: {output.strip()}")
    print("="*60)

    # 5. Output Table
    print_table(changes)

if __name__ == "__main__":
    load_sentiment_model()

    tests_to_run = []

    if BATCH_MODE == 1:
        print("=== RUNNING BATCH 1: SIMPLE TESTS ===")
        tests_to_run = BATCH_SIMPLE
    elif BATCH_MODE == 2:
        print("=== RUNNING BATCH 2: EXTENDED TESTS ===")
        tests_to_run = BATCH_COMPLEX
    elif BATCH_MODE == 3:
        print("=== RUNNING ALL TESTS ===")
        tests_to_run = BATCH_SIMPLE + BATCH_COMPLEX

    for i, test in enumerate(tests_to_run):
        print(f"\n--- Test {i+1}: {test['title']} ---")
        run_pessimizer(test['text'])
        time.sleep(1) # Pause for clarity